### Athlete Country — Coverage Analysis

- 17 countries in raw data could not be matched against our country reference data
(see breakdown below)
- `XXX`: 1,783 rows — likely placeholder for unknown country
- Several typos identified with low row counts (≤ 27 rows each)

---

## Summary — Actions Required for Cleaning

### Standardization
- `Athlete country`: fix typos in Silver:
  - `swe` → `SWE`
  - `Ned` → `NED`
  - `SVE` → `SWE`
  - `GRB` → `GBR`
  - `IRE` → `IRL`

### Null handling
- `Athlete country`: replace `XXX` with `"Unknown"`

In [0]:
# Read in historic data and country referq data from bronze layer tables

df_raw = spark.read.table("marathos.bronze.um_races_raw")
df_countries = spark.read.table("marathos.bronze.countries_raw")

In [0]:
# Unique countries in historic data
countries_in_dataset = set(
    df_raw.select("Athlete country")
    .distinct()
    .toPandas()["Athlete country"]
)

# Unique countries in reference data
countries_in_ref = set(
        df_countries.select("IOC")
        .distinct()
        .toPandas()["IOC"]
)

# Countries that can't be found in reference data
df_missing_countries = countries_in_dataset.difference(countries_in_ref)
print(f"Nr of missing countries in referance data: {len(df_missing_countries)}")
print(f"Missing countries: {df_missing_countries}")


In [0]:
from pyspark.sql.functions import col

# Check nr of athletes per missing country

df_raw.filter(
    col("Athlete country").isin(list(df_missing_countries))
).groupBy("Athlete country") \
    .count() \
    .orderBy("count", ascending=False) \
    .display()